# 06 · Tuning, operating point, capacity analysis, and the single-touch test evaluation

> Educational decision-support prototype trained on synthetic PaySim data. Outputs are risk scores and review priorities that help human investigators decide what to review first. This system makes no fraud or AML determination and performs no automatic blocking, account closure, customer risk rating, or regulatory reporting. Results on synthetic data do not establish real-world detection effectiveness, fairness, or regulatory suitability.

This notebook **reads saved artifacts only**. It never calls `evaluate --split test`; the test split is scored exactly once by the CLI and any re-evaluation is recorded with a reason in `data/processed/test_access.json`.

In [ ]:
from pathlib import Path

import json
import pandas as pd
import yaml
from IPython.display import Markdown, display

from aml_triage.config import load

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load(ROOT / "configs" / "base.yaml")
REPORTS = ROOT / cfg.paths.reports_dir
RUNS = ROOT / cfg.paths.models_dir / "runs"
PROCESSED = ROOT / cfg.paths.processed_dir
print("config hash:", cfg.config_hash())

## Test-access state (must read `evaluated` with an empty `reevaluations` list)

In [ ]:
p = PROCESSED / "test_access.json"
state = json.loads(p.read_text()) if p.exists() else {"state": "locked"}
display(pd.Series(state).to_frame("value"))

## Tuned parameters

In [ ]:
rows = []
for f in sorted((ROOT / "configs" / "models").glob("*.tuned.yaml")):
    t = yaml.safe_load(f.read_text())
    rows.append({"model": t["id"], **{f"cv_{k}": v for k, v in t["tuned_on"].items() if k in ("best_cv_score", "seconds", "subsample_rows", "n_iter")}, "params": t["params"]})
display(pd.DataFrame(rows) if rows else Markdown("_No tuned configs yet._"))

## Operating point (chosen on validation, frozen before the test split was touched)

In [ ]:
op_path = ROOT / cfg.operating_point_path
display(Markdown(f"```yaml\n{op_path.read_text()}\n```") if op_path.exists() else Markdown("_Not chosen yet._"))

## Selection matrix

In [ ]:
p = REPORTS / "selection_matrix.md"
display(Markdown(p.read_text()) if p.exists() else Markdown("_Run `select` first._"))

## Capacity analysis

In [ ]:
p = REPORTS / "capacity_analysis.md"
if p.exists():
    display(Markdown(p.read_text().replace("](figures/", f"]({REPORTS.as_posix()}/figures/")))
else:
    display(Markdown("_Not written yet (task T059)._"))

## Model comparison (validation and test sections)

In [ ]:
p = REPORTS / "model_comparison.md"
if p.exists():
    display(Markdown(p.read_text().replace("](figures/", f"]({REPORTS.as_posix()}/figures/")))
else:
    display(Markdown("_Run `compare` first._"))

## Model card of the released bundle

In [ ]:
latest = ROOT / cfg.paths.models_dir / "LATEST"
if latest.exists():
    v = latest.read_text().strip()
    display(Markdown((ROOT / cfg.paths.models_dir / v / "model_card.md").read_text()))
else:
    display(Markdown("_No bundle released yet._"))